# 📡 TV Attribution: Campaign-Level Causal Impact Analysis

This notebook uses the Bayesian Structural Time Series (BSTS) model via the `tfcausalimpact` package to measure the aggregate campaign lift. We utilize a correlated control market to construct a counterfactual baseline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from causalimpact import CausalImpact

plt.style.use('seaborn-v0_8-whitegrid')

# Aggregate to Hourly to reduce BSTS noise and improve speed
sessions = pd.read_csv('../data/sessions.csv', parse_dates=['timestamp']).set_index('timestamp')
control = pd.read_csv('../data/control_market.csv', parse_dates=['timestamp']).set_index('timestamp')

hourly_sessions = sessions.resample('H').sum()
hourly_control = control.resample('H').sum()

# Merge for CausalImpact: first column is target (treated), second is control
df = pd.merge(hourly_sessions, hourly_control, on='timestamp', suffixes=('_treated', '_control'))
df.head()

### 1. Model Configuration

We select a central 2-week burst period to analyze. The pre-period is used for counterfactual model training.

In [ ]:
# Pre-period: Weeks 1-6
# Post-period (Burst): Weeks 7-8
pre_period = [df.index[0], df.index[1007]] # ~6 weeks
post_period = [df.index[1008], df.index[1343]] # ~2 weeks

ci = CausalImpact(df, pre_period, post_period)
print("CausalImpact model initialized.")

### 2. Campaign Lift Summary

Quantifying the total incremental conversion lift with 95% credible intervals.

In [ ]:
print(ci.summary())
print(ci.summary(output='report'))

### 3. Causal Graphs

Visualizing the counterfactual vs. observed data and cumulative impact.

In [ ]:
ci.plot()
plt.show()